In [1]:
# IndicWG 2025 — Hindi Word Grouping (BIO)
# Strategy: Token classification with BIO labels (B/I inside group, O outside),
# word-level decoding → groups joined with '__'.
#
# Note: Run cells top-to-bottom. Train, then generate predictions.csv.
#%pip install -q transformers torch pandas scikit-learn

import os, re, json, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, Trainer, TrainingArguments, set_seed
)

# ---- Config ----
model_name = 'ai4bharat/IndicBERTv2-MLM-only'  # or 'ai4bharat/indic-bert'
num_epochs = 20
lr = 3e-5
seed = 42
set_seed(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [2]:
# Load train.csv and optional dev.csv
full_train = pd.read_csv('train.csv')
dev = pd.read_csv('dev.csv') if os.path.exists('dev.csv') else pd.DataFrame(columns=['Input Sentence'])

# 90:10 train/val split from train.csv
train_df, val_df = train_test_split(full_train, test_size=0.1, random_state=seed, shuffle=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print('Train:', len(train_df), 'Val:', len(val_df), 'Dev:', len(dev))


Train: 495 Val: 55 Dev: 100


In [3]:
# Build BIO labels from (Input Sentence, Output Sentence)
def make_bio(input_text: str, output_text: str):
    inp = str(input_text).strip()
    out = str(output_text).strip()
    input_tokens = inp.split()
    labels = []
    if out == '' and inp != '':
        # No labels provided → treat as all 'O'
        return input_tokens, ['O'] * len(input_tokens)
    for group in out.split():
        parts = group.split('__')
        if len(parts) == 1:
            labels.append('O')
        else:
            for j in range(len(parts)):
                labels.append('B' if j == 0 else 'I')
    # Length guard
    if len(labels) != len(input_tokens):
        # Fallback: keep data point by using all 'O' to avoid crashes
        labels = ['O'] * len(input_tokens)
    return input_tokens, labels

def build_texts_labels(df: pd.DataFrame, expect_labels=True):
    texts, labs = [], []
    for i in range(len(df)):
        inp = df.iloc[i]['Input Sentence']
        if expect_labels:
            out = df.iloc[i]['Output Sentence']
        else:
            out = ''
        tokens, labels = make_bio(inp, out)
        texts.append(tokens)
        labs.append(labels)
    return texts, labs

train_texts, train_labels = build_texts_labels(train_df, expect_labels=True)
val_texts, val_labels = build_texts_labels(val_df, expect_labels=True)
dev_texts, dev_labels = build_texts_labels(dev, expect_labels=('Output Sentence' in dev.columns))
print('Example:', list(zip(train_texts[0][:10], train_labels[0][:10])))


Example: [('मिस्टर', 'B'), ('कॉस्टेलो', 'I'), ('ने', 'I'), ('कहा', 'O'), ('कि', 'O'), ('जब', 'O'), ('परमाणु', 'B'), ('ऊर्जा', 'I'), ('उत्पादन', 'I'), ('आर्थिक', 'O')]


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
label2id = {'B': 0, 'I': 1, 'O': 2}
id2label = {v: k for k, v in label2id.items()}

def encode_data(texts, labels):
    enc = tokenizer(texts, is_split_into_words=True, truncation=True, padding=True)
    encoded_labels = []
    for i, lab in enumerate(labels):
        word_ids = enc.word_ids(batch_index=i)
        lab_ids = []
        for w in word_ids:
            if w is None:
                lab_ids.append(-100)
            else:
                lab_ids.append(label2id.get(lab[w], 2))  # default 'O'
        encoded_labels.append(lab_ids)
    enc['labels'] = encoded_labels
    return enc

train_enc = encode_data(train_texts, train_labels)
val_enc = encode_data(val_texts, val_labels)
dev_enc = encode_data(dev_texts, dev_labels)

import torch.utils.data as tud
class GroupDataset(tud.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

train_dataset = GroupDataset(train_enc)
val_dataset = GroupDataset(val_enc)
dev_dataset = GroupDataset(dev_enc)


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

c:\Users\MANAV DHAMECHA\miniconda3\envs\gec\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MANAV DHAMECHA\.cache\huggingface\hub\models--ai4bharat--IndicBERTv2-MLM-only. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fall

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [5]:
# Model + weighted loss to reduce 'all-O' bias
num_labels = 3
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
)
data_collator = DataCollatorForTokenClassification(tokenizer)

# Compute class weights from train labels
from collections import Counter
flat = [t for seq in train_labels for t in seq]
cnt = Counter(flat)
total = sum(cnt.values())
def inv_freq(c):
    return max(1.0, total / max(1, c))
wB, wI, wO = inv_freq(cnt.get('B', 1)), inv_freq(cnt.get('I', 1)), inv_freq(cnt.get('O', 1))
s = wB + wI + wO
class_weights = torch.tensor([wB/s, wI/s, wO/s], dtype=torch.float)

from transformers import Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**{k: v for k, v in inputs.items() if k != 'labels'})
        logits = outputs.get('logits')
        cw = class_weights.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=cw, ignore_index=-100)
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

try:
    args = TrainingArguments(
        output_dir='./results',
        evaluation_strategy='epoch',
        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        save_total_limit=2,
        logging_steps=50,
        report_to='none',
    )
except TypeError:
    print('Falling back to compatible TrainingArguments without evaluation_strategy')
    args = TrainingArguments(
        output_dir='./results',
        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        save_total_limit=2,
        logging_steps=50,
        report_to='none',
    )

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)
print('✅ Trainer ready (weighted loss)')


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-only and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Falling back to compatible TrainingArguments without evaluation_strategy


C:\Users\MANAV DHAMECHA\AppData\Local\Temp\ipykernel_83944\1899472827.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


✅ Trainer ready (weighted loss)


In [6]:
# Train (uncomment to run)
trainer.train()
#print('Run trainer.train() to fine-tune the model')


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Step,Training Loss
50,0.821000
100,0.376700
150,0.228000
200,0.171300
250,0.118600
300,0.082700
350,0.059800
400,0.037300
450,0.033900
500,0.031800


TrainOutput(global_step=1240, training_loss=0.08438785356138984, metrics={'train_runtime': 860.9142, 'train_samples_per_second': 11.499, 'train_steps_per_second': 1.44, 'total_flos': 1192381370848800.0, 'train_loss': 0.08438785356138984, 'epoch': 20.0})

In [7]:
# Utilities: decode token-level predictions → grouped sentences
def decode_to_word_labels(pred_ids, texts, tokenizer):
    tokenized = tokenizer(texts, is_split_into_words=True, truncation=True, padding=True)
    word_level = []
    for i in range(len(texts)):
        word_ids = tokenized.word_ids(batch_index=i)
        tok_preds = pred_ids[i]
        labels = []
        seen = set()
        for ti, wi in enumerate(word_ids):
            if wi is None or wi in seen:
                continue
            labels.append(int(tok_preds[ti]))
            seen.add(wi)
        word_level.append(labels)
    return word_level

def reconstruct_sentence(words, word_label_ids, id2label):
    # Convert ids→BIO labels
    labs = [id2label.get(int(x), 'O') for x in word_label_ids]
    out, i = [], 0
    while i < len(words):
        lab = labs[i] if i < len(labs) else 'O'
        if lab == 'B':
            group = [words[i]]
            i += 1
            while i < len(words):
                lab_i = labs[i] if i < len(labs) else 'O'
                if lab_i == 'I':
                    group.append(words[i])
                    i += 1
                else:
                    break
            out.append('__'.join(group))
        else:
            out.append(words[i])
            i += 1
    return ' '.join(out)


In [8]:
# Predict on dev and write predictions.csv
import numpy as np

if len(dev_texts) == 0:
    print('No dev samples found; skipping predictions.csv')
else:
    pred_output = trainer.predict(dev_dataset)
    preds = np.argmax(pred_output.predictions, axis=-1)
    word_level = decode_to_word_labels(preds, dev_texts, tokenizer)
    outputs = []
    for words, wlabs in zip(dev_texts, word_level):
        outputs.append(reconstruct_sentence(words, wlabs, id2label))
    out_df = pd.DataFrame({
        'Input Sentence': dev.get('Input Sentence', pd.Series(['']*len(outputs))),
        'Output Sentence': outputs,
    })
    out_df.to_csv('predictions.csv', index=False, encoding='utf-8')
    print(f'✅ Saved {len(out_df)} rows to predictions.csv')


✅ Saved 100 rows to predictions.csv


In [9]:
# Evaluation on validation split (Exact Match)
import numpy as np, os, json
val_pred_output = trainer.predict(val_dataset)
val_preds = np.argmax(val_pred_output.predictions, axis=-1)
val_word_level = decode_to_word_labels(val_preds, val_texts, tokenizer)
val_outputs = [reconstruct_sentence(w, l, id2label) for w, l in zip(val_texts, val_word_level)]
gold = val_df['Output Sentence'].astype(str).tolist()
pred = val_outputs
total = len(gold)
correct = sum(1 for g, p in zip(gold, pred) if p == g)
acc = (correct/total)*100.0 if total else 0.0
print('✅ Exact Match on val:', f'{acc:.4f}% (', correct, '/', total, ')')
os.makedirs('evaluation_output', exist_ok=True)
with open('evaluation_output/scores.txt', 'w', encoding='utf-8') as f:
    f.write(f'exact_match: {acc:.6f}\n')
with open('evaluation_output/scores.json', 'w', encoding='utf-8') as f:
    json.dump({'exact_match': acc}, f, ensure_ascii=False)


✅ Exact Match on val: 52.7273% ( 29 / 55 )


In [10]:
# (Optional) Inference from a saved checkpoint
# Set CHECKPOINT_DIR to a specific checkpoint under ./results/ if needed
# CHECKPOINT_DIR = './results/checkpoint-XXXX'
# tok2 = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
# mdl2 = AutoModelForTokenClassification.from_pretrained(CHECKPOINT_DIR)
# tr2 = Trainer(model=mdl2, tokenizer=tok2)
# Use tr2.predict(...) + same decoding utilities above
print('Inference helper ready (set CHECKPOINT_DIR and uncomment)')


Inference helper ready (set CHECKPOINT_DIR and uncomment)
